# 02 · Baseline Results
Before running: `02_build_features.py` → `03_train_baseline.py`

In [ ]:
import sys, json; sys.path.insert(0, "../src")
import pandas as pd, matplotlib.pyplot as plt
from wellsentinel.config import load_config
from wellsentinel.data import EVENT_NAMES
cfg = load_config(); R = cfg["paths"]["reports_dir"]
m = json.load(open(R / "metrics_baseline.json"))
print("macro-F1:", round(m["window"]["macro_f1"], 3)); m["event"]

In [ ]:
pd.Series(m["per_class_f1"]).sort_values().plot.barh(figsize=(7, 4), title="F1 per event (unseen wells)"); plt.xlim(0, 1)

In [ ]:
ev = pd.read_csv(R / "event_detection_test.csv")
ev.groupby("event").agg(instances=("instance_id", "size"), detected=("detected", "mean"),
                        early=("early_warning", "mean"), median_delay_min=("delay_min", "median")).round(2)

In [ ]:
from IPython.display import Image
Image(str(cfg["paths"]["figures_dir"] / "confusion_matrix.png"))

## Feature / sensor importance (permutation)

In [ ]:
from wellsentinel.models import load_bundle
from wellsentinel.split import train_test_masks
from wellsentinel.explain import permutation_ranking, sensor_ranking
b = load_bundle(cfg["paths"]["models_dir"] / "baseline.joblib")
table = pd.read_parquet(cfg["paths"]["processed_dir"] / "features.parquet")
folds = pd.read_csv(cfg["paths"]["interim_dir"] / "inventory.csv")
_, te = train_test_masks(table, folds, m["test_fold"])
rank = permutation_ranking(b["model"], table.loc[te, b["feature_cols"]], table.loc[te, "target"], max_rows=3000)
sensor_ranking(rank).head(12).plot.barh(x="sensor", y="importance", figsize=(7, 4), legend=False).invert_yaxis()

## Takeaways → copy to `docs/WORK_LOG.md`
- ...
- ...